In [0]:
# =============================================================================
# ICEBASE — PHASE 4 | NOTEBOOK 02
# ML Model 1 — K-Means Fan Segmentation
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# STANDALONE NOTEBOOK — Run on a DEDICATED (single-user) cluster.
#
# WHAT THIS NOTEBOOK DOES:
#   Trains a K-Means clustering model on the fan feature table to produce
#   5 fan segments. Each cluster is labeled with a business-meaningful name
#   based on its behavioral profile. Segment assignments are written back
#   to icebase.gold.ml_fan_segments for use in dashboards and campaigns.
#
# MODEL DETAILS:
#   Algorithm:    K-Means (sklearn)
#   Features:     RFM + behavioral signals from icebase.gold.fan_features
#   Clusters:     5 (validated with Elbow method + Silhouette score)
#   Tracking:     MLflow experiment — all runs logged with metrics
#   Registry:     Best model registered to icebase.gold.kmeans_fan_segmentation
#
# SEGMENT LABELS (assigned post-hoc based on cluster profiles):
#   Season Core    : High frequency, high spend, low recency, low promo
#   High Value New : High spend, low tenure, low promo, good recency
#   Casual Fan     : Moderate frequency, price sensitive, medium recency
#   Promo Hunter   : High promo rate/sensitivity, lower net spend
#   Lapsed         : High recency (days since last), low frequency
#
# OUTPUT TABLE:
#   icebase.gold.ml_fan_segments
#   Columns: customer_id, segment_id, segment_label, distance_to_centroid,
#            scored_at, model_version
# =============================================================================

In [0]:
# COMMAND ----------
# ── CELL 1: Imports ────────────────────────────────────────────────────────
 
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from mlflow.models import infer_signature
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
from pyspark.sql import functions as F
 
mlflow.set_registry_uri("databricks-uc")
 
CATALOG         = "icebase"
GOLD            = f"{CATALOG}.gold"
FEATURE_TABLE   = f"{GOLD}.fan_features"
SEGMENT_TABLE   = f"{GOLD}.ml_fan_segments"
EXPERIMENT_NAME = "/Users/{}/icebase/phase4_segmentation"
REGISTERED_MODEL= f"{GOLD}.kmeans_fan_segmentation"
 
# Set MLflow experiment — creates it if it doesn't exist
import re
username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/icebase/phase4_segmentation")
 
print("✅ Segmentation notebook loaded")
print(f"   Feature table:    {FEATURE_TABLE}")
print(f"   Output table:     {SEGMENT_TABLE}")
print(f"   Registered model: {REGISTERED_MODEL}")

In [0]:
# COMMAND ----------
# ── CELL 2: Load Features ──────────────────────────────────────────────────
#
# Load the fan feature table as a Pandas DataFrame for sklearn.
# Exclude the churn_flag — that's a supervised label, not a clustering input.
# Exclude binary event flags from clustering (too sparse, distort geometry).
 
CLUSTER_FEATURES = [
    "recency_days",
    "frequency_games",
    "monetary_net",
    "promo_sensitivity",
    "advance_purchase_rate",
    "avg_seat_tier_rank",
    "tenure_days",
    "is_season_holder",
    "channels_used",
]
 
fan_features_df = spark.table(FEATURE_TABLE).toPandas()
 
X = fan_features_df[CLUSTER_FEATURES].fillna(0)
customer_ids = fan_features_df["customer_id"].values
 
print(f"✅ Features loaded: {X.shape[0]:,} fans × {X.shape[1]} features")
print(f"   Features used: {CLUSTER_FEATURES}")

In [0]:
# COMMAND ----------
# ── CELL 3: Elbow Method — Find Optimal K ─────────────────────────────────
#
# Run K-Means for k=2 through k=8, log inertia and silhouette score for each.
# This is how you justify the choice of 5 clusters with data.
# All runs are logged to MLflow so you can compare in the Experiments UI.
 
print("Running Elbow analysis (k=2 to 8)...")
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
elbow_results = []
 
with mlflow.start_run(run_name="elbow_analysis"):
    for k in range(2, 9):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        inertia  = km.inertia_
        sil_score = silhouette_score(X_scaled, labels, sample_size=2000, random_state=42)
 
        elbow_results.append({"k": k, "inertia": inertia, "silhouette": sil_score})
        mlflow.log_metric("inertia",   inertia,   step=k)
        mlflow.log_metric("silhouette", sil_score, step=k)
        print(f"  k={k}: inertia={inertia:,.0f}  silhouette={sil_score:.4f}")
 
    # Log elbow plot as artifact
    ks         = [r["k"]         for r in elbow_results]
    inertias   = [r["inertia"]   for r in elbow_results]
    silhouettes= [r["silhouette"] for r in elbow_results]
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(ks, inertias, "bo-");  ax1.set_title("Elbow — Inertia");  ax1.set_xlabel("k")
    ax2.plot(ks, silhouettes, "ro-"); ax2.set_title("Silhouette Score"); ax2.set_xlabel("k")
    plt.tight_layout()
    plt.savefig("/tmp/elbow_plot.png", dpi=100)
    mlflow.log_artifact("/tmp/elbow_plot.png")
    plt.close()
 
print("\n✅ Elbow analysis complete — check MLflow Experiments UI for the plot")
print("   Proceed with k=5 based on elbow inflection and business segment count")

In [0]:
# COMMAND ----------
# ── CELL 4: Train Final K-Means Model (k=5) ───────────────────────────────
#
# Train the production model with k=5 clusters.
# Use a sklearn Pipeline (Scaler + KMeans) so scaling is bundled with
# the model — prevents training/serving skew at inference time.
# Log to MLflow and register in Unity Catalog.
 
K = 5
 
with mlflow.start_run(run_name=f"kmeans_k{K}_final") as run:
 
    # ── Train ──────────────────────────────────────────────────────────
    pipeline = Pipeline([
        ("scaler",  StandardScaler()),
        ("kmeans",  KMeans(n_clusters=K, random_state=42, n_init=10, max_iter=300)),
    ])
    pipeline.fit(X)
 
    labels    = pipeline.predict(X)
    X_scaled  = pipeline.named_steps["scaler"].transform(X)
    sil_score = silhouette_score(X_scaled, labels, sample_size=2000, random_state=42)
    inertia   = pipeline.named_steps["kmeans"].inertia_
 
    # ── Log parameters and metrics ────────────────────────────────────
    mlflow.log_params({
        "k":           K,
        "n_init":      10,
        "max_iter":    300,
        "random_state":42,
        "features":    ",".join(CLUSTER_FEATURES),
        "n_samples":   len(X),
    })
    mlflow.log_metrics({
        "silhouette_score": round(sil_score, 4),
        "inertia":          round(inertia, 2),
    })
 
    # ── Infer model signature ──────────────────────────────────────────
    input_sample  = X.iloc[:5]
    output_sample = pd.DataFrame({"segment_id": pipeline.predict(input_sample)})
    signature     = infer_signature(input_sample, output_sample)
 
    # ── Log and register model ─────────────────────────────────────────
    mlflow.sklearn.log_model(
        sk_model              = pipeline,
        artifact_path         = "model",
        signature             = signature,
        input_example         = input_sample,
        registered_model_name = REGISTERED_MODEL,
    )
 
    run_id = run.info.run_id
    print(f"✅ Model trained and registered")
    print(f"   Silhouette score: {sil_score:.4f}  (higher is better, max=1.0)")
    print(f"   Inertia:          {inertia:,.0f}")
    print(f"   Run ID:           {run_id}")
    print(f"   Registered as:    {REGISTERED_MODEL}")

In [0]:
# COMMAND ----------
# ── CELL 5: Label Clusters with Business Names ─────────────────────────────
#
# K-Means assigns numeric cluster IDs (0–4). We need to map these to
# meaningful business segment labels by profiling each cluster.
# The mapping is done by inspecting cluster centroids and median feature values.
#
# This step MUST be done manually after reviewing the cluster profiles below.
# The labels in SEGMENT_LABELS may need to be adjusted based on what
# the actual clusters look like in your data run.
 
# Get cluster assignments and add to features DataFrame
fan_features_df["segment_id"]     = labels
fan_features_df["distance_to_centroid"] = np.min(
    pipeline.named_steps["kmeans"].transform(
        pipeline.named_steps["scaler"].transform(X)
    ),
    axis=1
)
 
# Profile each cluster
print("── Cluster Profiles (review these to assign labels) ──")
cluster_profile = (
    fan_features_df.groupby("segment_id")[
        CLUSTER_FEATURES + ["churn_flag"]
    ]
    .median()
    .round(3)
)
print(cluster_profile.to_string())
print()
 
# ─────────────────────────────────────────────────────────────────────────
# MANUAL STEP: Review the cluster profiles printed above and update the
# SEGMENT_LABELS dictionary to match your actual cluster behavior.
# The keys are cluster IDs (0–4), values are the business segment names.
#
# Assignment guide:
#   Season Core   → lowest recency_days, highest frequency_games, low promo_sensitivity
#   High Value New→ low recency, moderate-high monetary, low tenure
#   Casual Fan    → moderate recency, low-moderate frequency, moderate promo
#   Promo Hunter  → highest promo_sensitivity, lowest monetary_net
#   Lapsed        → highest recency_days, lowest frequency_games
# ─────────────────────────────────────────────────────────────────────────
 
# Default mapping — REVIEW AND ADJUST based on cluster profile output above
SEGMENT_LABELS = {
    0: "Promo Hunter",
    1: "Lapsed",
    2: "Season Core",
    3: "Casual Fan",
    4: "Deeply Lapsed",
}
 
fan_features_df["segment_label"] = fan_features_df["segment_id"].map(SEGMENT_LABELS)
 
print("── Segment counts ──")
print(fan_features_df["segment_label"].value_counts().to_string())

In [0]:
# COMMAND ----------
# ── CELL 6: Write Segment Scores to Gold ──────────────────────────────────
 
from datetime import datetime
 
segments_pdf = fan_features_df[
    ["customer_id", "segment_id", "segment_label", "distance_to_centroid"]
].copy()
segments_pdf["scored_at"]      = datetime.now()
segments_pdf["model_version"]  = "kmeans_k5_v1"
segments_pdf["model_name"]     = REGISTERED_MODEL
 
segments_sdf = spark.createDataFrame(segments_pdf)
 
(
    segments_sdf
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SEGMENT_TABLE)
)
 
count = spark.table(SEGMENT_TABLE).count()
print(f"✅ ml_fan_segments written | {count:,} rows")

In [0]:
# COMMAND ----------
# ── CELL 7: Segment Business Validation ───────────────────────────────────
#
# Join segments back to customer_360 to validate the business story:
# Does Promo Hunter actually have high promo_sensitivity?
# Does Season Core actually have the highest spend?
 
print("── Segment validation — business metrics per segment ──")
display(
    spark.table(SEGMENT_TABLE)
    .join(spark.table(f"{GOLD}.customer_360"), "customer_id", "left")
    .groupBy("segment_label")
    .agg(
        F.count("*")                                .alias("fan_count"),
        F.round(F.avg("total_spend"), 2)            .alias("avg_total_spend"),
        F.round(F.avg("games_attended"), 1)         .alias("avg_games_attended"),
        F.round(F.avg("days_since_last"), 0)        .alias("avg_days_since_last"),
        F.round(F.avg("promo_sensitivity"), 3)      .alias("avg_promo_sensitivity"),
        F.round(F.avg("tenure_days"), 0)            .alias("avg_tenure_days"),
        F.sum(F.col("is_churn_risk").cast("int"))   .alias("churn_risk_count"),
    )
    .orderBy(F.col("avg_total_spend").desc())
)
 
# EXPECTED:
#   Season Core    → highest avg_total_spend, lowest avg_days_since_last
#   Promo Hunter   → highest avg_promo_sensitivity
#   Lapsed         → highest avg_days_since_last, highest churn_risk_count